# Create an elastic network for 6o2h (lysozyme, p1 space group) and compute diffuse scattering from it.

Goal: use the ENM for triclinic lysozyme and generate a diffuse map.

Code mostly lifted from 20260812_lysozyme-tri-refinement.ipynb.

Steps:

1. Use ENM for 6o2h previously generated with refined spring constants from the MATLAB iteration.
2. Generate diffuse intensity as a function of (h,k,l).

In [1]:
#I will use gemmi to parse the PDB file 6o2h.cif.
#I then want to go towards constructing a Hessian for the rigid body ENM with springs organized nicely in a dataframe.
from goodvibes import enm, dynamat
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import gemmi
import random
from scipy import sparse
from collections import defaultdict

# 1. Assign rigid groups

In [2]:
#A lot of these lines will be borrowed from sandbox/steve/proto-hessian.ipynb

# !cd test_data && curl -O https://files.rcsb.org/download/6O2H.cif

coordinate_file = 'test_data/6O2H.cif'

st = gemmi.read_structure(coordinate_file)
st.setup_entities()  # supposed to be good practice

print('Unit Cell:',st.cell.parameters)
print('Space Group:',st.spacegroup_hm)

groups = [
    gemmi.Selection('//A;polymer'),
    gemmi.Selection('//B;polymer'),
]

tls_groups = []

# calculate the group center of mass, store as TLS group origin
for group_id, selection in enumerate(groups):
    model = selection.copy_model_selection(st[0])
    com = model.calculate_center_of_mass()
    g = gemmi.TlsGroup()
    g.id = f"TLS{group_id}"  # why is this a string? why can't I access num_id property?
    g.origin = com
    tls_groups.append(g)

st.meta.refinement[0].tls_groups = tls_groups

for group_id, selection in enumerate(groups):
    for model in selection.models(st):
        for chain in selection.chains(model):
            for residue in selection.residues(chain):
                for atom in selection.atoms(residue):
                    atom.tls_group_id = group_id

Unit Cell: (27.424, 32.134, 34.513, 88.657, 108.46, 111.877)
Space Group: P 1


# 2. and 3. Pack unit cell and construct contact list

In [3]:
enm._pack_unit_cell(st, inplace=True)

df = enm._find_contacts(
   st,
   distance_cutoff=4.0,
   include_h=False,
)

def address_from_cra_string(cra):
    chain, residue, atom = cra.split('/')
    resname, seqid = residue.split()
    if '.' in atom:
        atom, altloc = atom.split('.')
    else:
        altloc = '\x00'
    addr =  gemmi.AtomAddress(chain, gemmi.SeqId(seqid), resname, atom, altloc)
    return addr

df['group_id1'] = -1
df['group_id2'] = -1

# add group ID columns to dataframe
for row in df.itertuples():
    cra1 = st[0].find_cra(address_from_cra_string(row.cra1))
    cra2 = st[0].find_cra(address_from_cra_string(row.cra2))
    df.at[row.Index, 'group_id1'] = cra1.atom.tls_group_id
    df.at[row.Index, 'group_id2'] = cra2.atom.tls_group_id

df['external'] = False

# add a column that indicates whether the contact is external (between different groups)
for row in df.itertuples():
    if row.group_id1 == -1 or row.group_id2 == -1:
        df.at[row.Index, 'external'] = False
    elif (row.sym_idx1 == row.sym_idx2) and (row.pbc_shift1 == row.pbc_shift2):
        # same ASU, check if the two residues have the same flag (i.e. belong to the same rigid group)
        df.at[row.Index, 'external'] = (row.group_id1 != row.group_id2)
    else:
        # different ASU, any group flag (not null)
        df.at[row.Index, 'external'] = True

# create a new dataframe that contains only the external contacts, dropping the 'external' column, reset index.
external_df = df[df['external']].drop(columns=['external']).reset_index(drop=True)

# 4. Expand to fill unit cell and add node positions, keeping symmetry labels on bonds

In [4]:
external_df_expanded = enm._symmetry_expand(external_df, st.cell.images)
external_df_expanded = external_df_expanded.rename(columns={'index': 'spring_class'})

def transform_position(st, sym_idx, pbc_shift, pos):
    if sym_idx == 0:
        image_transform = gemmi.Transform()
    else:
        image_transform = st.cell.images[sym_idx - 1]
    pbc_transform = gemmi.Transform(gemmi.Mat33(), gemmi.Vec3(*pbc_shift))
    t = pbc_transform @ image_transform
    return st.cell.orthogonalize(gemmi.Fractional(t.apply(st.cell.fractionalize(pos))))

external_df_expanded['r1'] = pd.Series([None] * len(external_df_expanded), dtype=object)
external_df_expanded['r2'] = pd.Series([None] * len(external_df_expanded), dtype=object)

for row in external_df_expanded.itertuples():
    cra1 = st[0].find_cra(address_from_cra_string(row.cra1))
    cra2 = st[0].find_cra(address_from_cra_string(row.cra2))
    pos1 = transform_position(st, row.sym_idx1, row.pbc_shift1, cra1.atom.pos)
    pos2 = transform_position(st, row.sym_idx2, row.pbc_shift2, cra2.atom.pos)
    external_df_expanded.at[row.Index, 'r1'] = (pos1.x, pos1.y, pos1.z)
    external_df_expanded.at[row.Index, 'r2'] = (pos2.x, pos2.y, pos2.z)

# 5. Delete duplicate springs due to PBC

In [5]:
# Can detect duplicates by dropping the pbc_shift columns and finding unique edges.

external_df_expanded_pbc = external_df_expanded.copy()

#Sort based on cra and sym_idx
for row in external_df_expanded.itertuples():
    if (row.cra1, row.sym_idx1) > (row.cra2, row.sym_idx2):
        external_df_expanded_pbc.at[row.Index, 'cra1'] = row.cra2
        external_df_expanded_pbc.at[row.Index, 'sym_idx1'] = row.sym_idx2
        external_df_expanded_pbc.at[row.Index, 'pbc_shift1'] = row.pbc_shift2
        external_df_expanded_pbc.at[row.Index, 'group_id1'] = row.group_id2
        external_df_expanded_pbc.at[row.Index, 'r1'] = row.r2
        external_df_expanded_pbc.at[row.Index, 'cra2'] = row.cra1
        external_df_expanded_pbc.at[row.Index, 'sym_idx2'] = row.sym_idx1
        external_df_expanded_pbc.at[row.Index, 'pbc_shift2'] = row.pbc_shift1
        external_df_expanded_pbc.at[row.Index, 'group_id2'] = row.group_id1
        external_df_expanded_pbc.at[row.Index, 'r2'] = row.r1

external_df_expanded_pbc.drop_duplicates(subset=['cra1','cra2','sym_idx1','sym_idx2'],inplace=True)
external_df_expanded_pbc.reset_index(drop=True,inplace=True)

#swap sites 1 and 2 again so that cra1 is back inside the unit cell.
for row in external_df_expanded_pbc.itertuples():
    if row.pbc_shift1!=(0,0,0):
        external_df_expanded_pbc.at[row.Index, 'cra1'] = row.cra2
        external_df_expanded_pbc.at[row.Index, 'sym_idx1'] = row.sym_idx2
        external_df_expanded_pbc.at[row.Index, 'pbc_shift1'] = row.pbc_shift2
        external_df_expanded_pbc.at[row.Index, 'group_id1'] = row.group_id2
        external_df_expanded_pbc.at[row.Index, 'r1'] = row.r2
        external_df_expanded_pbc.at[row.Index, 'cra2'] = row.cra1
        external_df_expanded_pbc.at[row.Index, 'sym_idx2'] = row.sym_idx1
        external_df_expanded_pbc.at[row.Index, 'pbc_shift2'] = row.pbc_shift1
        external_df_expanded_pbc.at[row.Index, 'group_id2'] = row.group_id1
        external_df_expanded_pbc.at[row.Index, 'r2'] = row.r1

#symmetry groups w can be found just by checking which atoms are connected. Group by cra1 and cra2. Springs that are symmetry-related connect the same cra1 and cra2.
df_symmetry = external_df_expanded_pbc.copy()
#delete columns: cra1 is always in unit cell, n1, n2 extraneous for now.
df_symmetry = df_symmetry.drop(columns = ['pbc_shift1'])
#rename column: pbc_shift is the vector pointing from the unit cell of cra1 to the unit cell of cra2.
df_symmetry = df_symmetry.rename(columns={'pbc_shift2':'pbc_shift'})
#reindex symmetry classes after dropping duplicates
mapping = {x: i for i, x in enumerate(sorted(df_symmetry['spring_class'].unique()))}
df_symmetry['spring_class'] = df_symmetry['spring_class'].map(mapping)
#count of unique bonds
symm_classes = df_symmetry.sort_values(by=['spring_class']).iloc[[-1]]['spring_class'].tolist()[0]+1

In [6]:
springs_refinement = np.load('krefinement.npy')

In [7]:
# 1. build df_atoms via space-group expansion
df_atoms = dynamat.build_df_atoms(st, groups)

# 2. rigid-body geometry: CM, generalized mass matrix, Cholesky
body_index, CM, L = dynamat.build_rigid_bodies(df_atoms)
M = len(body_index)
Linv = np.linalg.inv(L)

# 3. lattice / reciprocal lattice
lattice = dynamat.get_lattice(st.cell)
recip = dynamat.reciprocal_lattice(lattice)

# 4. df_symmetry already built elsewhere in pipeline
# df_symmetry = ...

#Based on units of spring constants:
steve_factor = 10.0*np.sqrt(4.11/1.66)/(2*np.pi) #~15.7/2pi

sparse_L, sparse_T = dynamat.accumulate_sparse_blocks(df_symmetry, body_index, CM, lattice)

classes = sorted(df_symmetry['spring_class'].unique())
# Grid should be fine enough for a smooth DOS -- 32^3 is a reasonable starting point; refine as needed.
grid_shape = (64,64,64)
sparse_blocks = dynamat.accumulate_sparse_grid_blocks(df_symmetry, body_index, CM, lattice, grid_shape)
# Get spring constants after refinement has been completed.
kL = {w: springs_refinement[-1].T[1][w] for w in classes}
kT = {w: springs_refinement[-1].T[0][w] for w in classes}

#Find bands on a grid of points to compute dos.
D_grid = dynamat.build_D_grid_lowmem(sparse_blocks, kL, kT, L, grid_shape, dtype=np.float32)
#these things are omega^2.
eigs = dynamat.compute_all_eigs_from_grid(D_grid)            # (N1,N2,N3,6M)

# 8. DOS from the full grid, plotted alongside the path band structure
centers, dos = dynamat.compute_dos(eigs, n_bins=400, broaden_bins=2.0)
#centers for histogram are omega, so need *factor to convert \omega to \nu in THz.
centers*=steve_factor
#normalize dos by # of q points.
dos /= (grid_shape[0]*grid_shape[1]*grid_shape[2])
#dos changes when units change.
# \int_{\omega_1}^{\omega_2}\rho_1(\omega)d\omega = \int_{\nu_1}^{\nu_2}\rho_2(\nu)d\nu => \rho_2(\nu) = (1/factor)*\rho_1(\omega).
dos /= steve_factor